In [15]:
import pandas as pd
import folium
from folium import plugins
import json
from shapely import wkt as shapely_wkt
from shapely.geometry import shape
df = pd.read_parquet("/Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/WORLDSAR/studies/WP2-Matching/S1_NISAR/deliverable/nisar_s1_IW_matches.parquet")


# === SPECIFY ROW INDEX TO DISPLAY ===
row_to_display = 8  # Change this to view different matches
# ====================================

if row_to_display >= len(df):
    print(f"Error: Row {row_to_display} out of range. DataFrame has {len(df)} rows (0-{len(df)-1}).")
else:
    # Create base map centered on the selected row
    selected_row = df.iloc[row_to_display]
    center_lat = selected_row['nisar_centroid_lat']
    center_lon = selected_row['nisar_centroid_lon']

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=5,
        tiles='OpenStreetMap'
    )

    # Add additional tile layers
    folium.TileLayer('cartodbpositron', name='CartoDB Positron').add_to(m)
    folium.TileLayer('cartodbdark_matter', name='CartoDB Dark').add_to(m)

    # Color schemes for different overlaps
    def get_color(overlap):
        if overlap > 0.7:
            return '#2ecc71'  # green for high overlap
        elif overlap > 0.5:
            return '#f39c12'  # orange for medium overlap
        else:
            return '#e74c3c'  # red for low overlap

    # Add NISAR footprint for selected row only
    nisar_group = folium.FeatureGroup(name='NISAR Footprint', show=True)
    row = selected_row
    try:
        # Parse WKT to polygon
        geom = shapely_wkt.loads(row['nisar_wkt'])
        coords = [[lat, lon] for lon, lat in geom.exterior.coords]
        
        color = get_color(row['spatial_overlap'])
        
        popup_html = f"""
        <div style="width:300px">
            <h4>NISAR Scene (Row {row_to_display})</h4>
            <b>Scene:</b> {row['nisar_scene_name'][:50]}...<br>
            <b>Time:</b> {row['nisar_start_time']}<br>
            <b>Platform:</b> {row['nisar_platform']}<br>
            <b>Polarization:</b> {row['nisar_polarization']}<br>
            <b>Flight Dir:</b> {row['nisar_flight_direction']}<br>
            <hr>
            <h4>Matched S1</h4>
            <b>Name:</b> {row['s1_name'][:50]}...<br>
            <b>Mode:</b> {row['s1_mode']}<br>
            <b>Type:</b> {row['s1_product_type']}<br>
            <b>Time Diff:</b> {row['time_window_days']:.1f} days<br>
            <b>Spatial Overlap:</b> {row['spatial_overlap']:.2%}<br>
        </div>
        """
        
        folium.Polygon(
            locations=coords,
            color=color,
            weight=3,
            fill=True,
            fillColor=color,
            fillOpacity=0.4,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"NISAR Row {row_to_display} - Overlap: {row['spatial_overlap']:.1%}"
        ).add_to(nisar_group)
        
        # Add centroid marker
        folium.CircleMarker(
            location=[row['nisar_centroid_lat'], row['nisar_centroid_lon']],
            radius=6,
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.9,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip="NISAR Centroid"
        ).add_to(nisar_group)
        
    except Exception as e:
        print(f"Error processing NISAR footprint: {e}")

    # Add S1 footprint for selected row only
    s1_group = folium.FeatureGroup(name='Sentinel-1 Footprint', show=True)
    row = selected_row
    try:
        # Parse S1 footprint (GeoJSON format)
        s1_footprint = row['s1_footprint']
        if isinstance(s1_footprint, str):
            # Try to fix common JSON issues (single quotes to double quotes)
            s1_footprint = s1_footprint.replace("'", '"')
            s1_footprint = json.loads(s1_footprint)
        geom = shape(s1_footprint)
        coords = [[lat, lon] for lon, lat in geom.exterior.coords]
        
        color = '#3498db'  # blue for S1
        
        popup_html = f"""
        <div style="width:300px">
            <h4>Sentinel-1</h4>
            <b>Name:</b> {row['s1_name']}<br>
            <b>Date:</b> {row['s1_start_date']}<br>
            <b>Mode:</b> {row['s1_mode']}<br>
            <b>Type:</b> {row['s1_product_type']}<br>
            <hr>
            <h4>Match Info</h4>
            <b>NISAR Scene:</b> {row['nisar_scene_name'][:50]}...<br>
            <b>Time Diff:</b> {row['time_window_days']:.1f} days<br>
            <b>Spatial Overlap:</b> {row['spatial_overlap']:.2%}<br>
        </div>
        """
        
        folium.Polygon(
            locations=coords,
            color=color,
            weight=3,
            fill=True,
            fillColor=color,
            fillOpacity=0.3,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"S1 - {row['s1_name'][:30]}"
        ).add_to(s1_group)
        
    except Exception as e:
        print(f"Error processing S1 footprint:")
        print(f"  S1 Name: {row['s1_name']}")
        print(f"  Error: {type(e).__name__}: {e}")
        print(f"  Footprint type: {type(row['s1_footprint'])}")
        print(f"  Footprint preview: {str(row['s1_footprint'])[:100]}...")

    nisar_group.add_to(m)
    s1_group.add_to(m)

    # Add info box with match details
    info_html = f'''
    <div style="position: fixed; 
                top: 10px; left: 50px; width: 320px; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:12px; padding: 10px">
        <h4 style="margin-top:0">Row {row_to_display} Match Details</h4>
        <b>Spatial Overlap:</b> {selected_row['spatial_overlap']:.2%}<br>
        <b>Time Difference:</b> {selected_row['time_window_days']:.1f} days<br>
        <b>NISAR:</b> {selected_row['nisar_scene_name'][:40]}...<br>
        <b>S1:</b> {selected_row['s1_name'][:40]}...
    </div>
    '''
    m.get_root().html.add_child(folium.Element(info_html))

    # Add legend
    legend_html = '''
    <div style="position: fixed; 
                bottom: 50px; right: 50px; width: 200px; height: 150px; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:14px; padding: 10px">
        <h4 style="margin-top:0">Spatial Overlap</h4>
        <p><span style="color:#2ecc71">●</span> High (> 70%)</p>
        <p><span style="color:#f39c12">●</span> Medium (50-70%)</p>
        <p><span style="color:#e74c3c">●</span> Low (< 50%)</p>
        <p><span style="color:#3498db">●</span> Sentinel-1</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))

    # Add layer control
    folium.LayerControl().add_to(m)

    # Add fullscreen button
    plugins.Fullscreen().add_to(m)

    # Display the map
    display(m)